# Cycler metadata

`bdf.read()` returns the measurement table and a `Metadata` object. The plugin fills `Metadata` with
what the cycler wrote in the file header. You add what the header does not state, and `bdf.save()`
keeps both.

In [ ]:
import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from pydantic import ValidationError

import bdf
from bdf.plugins import PLUGINS

pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_hide_dataframe_shape(True);

## Read a file

The header timestamps of this BioLogic file state no time zone, so `tz` gives the zone of the lab.

In [ ]:
path = Path(
    "in/ingest/example_raw_data_collection/nacr32140mp10/"
    "SINTEF__hina-nacr32140mp10-202410-3gvjz6__20250617__2C__25degC.mpt"
)
df, meta = bdf.read(path, tz="Europe/Oslo")

`filled()` lists the fields that hold a value.

In [ ]:
def filled(meta):
    out = {}

    def walk(node, prefix):
        if isinstance(node, dict):
            for key, value in node.items():
                walk(value, f"{prefix}{key}.")
        elif node not in (None, [], {}):
            out[prefix[:-1]] = node

    walk(meta.model_dump(mode="json", exclude={"raw", "bdf"}), "")
    return out


pl.DataFrame({"field": list(filled(meta)), "value": [json.dumps(v) for v in filled(meta).values()]})

## How the plugin finds each value

A plugin holds one rule per field. A rule is a regex for a text header, or a key path for a binary
header, and a normalization that converts the match.

In [ ]:
pl.DataFrame(
    [
        {"field": ".".join(target.path), "regex": rule.pattern.pattern, "normalization": rule.normalization.kind}
        for target, rule in PLUGINS["biologic_mpt"].metadata_parser.rules
    ]
)

One rule, step by step:

In [ ]:
target, rule = next((t, r) for t, r in PLUGINS["biologic_mpt"].metadata_parser.rules if t.path[-1] == "started_at")
text = rule.extract(meta.raw.splitlines())
epoch = rule.normalization.scalar(text, tz="Europe/Oslo")
print(repr(text), "->", epoch, "->", datetime.fromtimestamp(epoch, timezone.utc))

`meta.raw` keeps the whole header, including the lines that no rule reads.

In [ ]:
print("\n".join(meta.raw.splitlines()[:16]))

## Add metadata

Assign to the records. Each assignment is validated against the BattINFO schema.

In [ ]:
cell = meta.battinfo_cell.cell_instance
cell.name = "hina-nacr32140mp10-202410-3gvjz6"
cell.batch_id = "202410"

test = meta.battinfo_test
test.license = "CC-BY-4.0"
test.test.kind = "cycling"
test.test.conditions = {"ambient_temperature": {"value": 25.0, "unit": "degC"}}

dataset = meta.battinfo_dataset.dataset
dataset.name = "Hina NaCR32140-MP10 sodium-ion cell test data"
dataset.license = "CC-BY-4.0"
dataset.keywords = ["na-ion", "NaCR32140-MP10", "cycling"]
dataset.creators = [
    {"name": "Simon Clark", "affiliation": {"name": "SINTEF"}, "same_as": "https://orcid.org/0000-0002-8758-6109"},
    {"name": "Julian Tolchard", "affiliation": {"name": "SINTEF"}, "same_as": "https://orcid.org/0000-0003-0934-4515"},
]

In [ ]:
try:
    test.test.kind = "OCV"
except ValidationError as exc:
    print(exc.errors()[0]["msg"])

## Save and read back

`bdf.save()` writes the metadata to a `.metadata.json` sidecar. `bdf.read()` of the saved file
restores it.

In [ ]:
artifact = Path(tempfile.mkdtemp()) / "hina-nacr32140mp10-202410-3gvjz6.bdf.parquet"
bdf.save(df, artifact, metadata=meta)

sidecar = artifact.with_suffix(".metadata.json")
print(json.dumps(json.loads(sidecar.read_text(encoding="utf-8"))["battinfo_test"]["test"], indent=2))

In [ ]:
_, restored = bdf.read(artifact)
assert filled(restored) == filled(meta)
assert restored.raw == meta.raw